# Factor risk models from returns alone

Two ways to build a factor risk model when all you have is a return panel:

- **Statistical (PCA / nonlinear shrinkage)** estimates both the factors and the
  loadings from the return covariance. No outside data, no interpretation.
- **Fama-French time-series** takes published factor *return* series (Kenneth
  French, free) as the factors, and estimates each name's loadings by regressing
  its returns on them. The loadings come from returns alone, and the factors have
  names (market, size, value, profitability, investment, momentum).

This notebook builds the Fama-French model and compares its covariance to the
sample, nonlinear shrinkage (QIS) and the PCA statistical model from
`covariance.ipynb`. The Fama-French factors are point-in-time safe by
construction (realised return series) and lag live data by about two months, so
the most recent sessions drop out.

In [ ]:
import io
import urllib.request
import zipfile

import numpy as np
import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go

from sdp.config import settings

wh = duckdb.connect(str(settings.warehouse_path), read_only=True)   # read-only: coexists with Jupyter

## The return panel

Liquid common stock (`in_universe`), full history over the window, so every
estimator sees the same T x N matrix. Same pull as `covariance.ipynb`.

In [ ]:
N_NAMES = 1000     # tickers
TOTAL = 780        # trailing sessions of prices

latest = wh.execute("select max(date) from main_staging.stg_universe").fetchone()[0]
names = wh.execute(f"""
    select ticker from main_staging.stg_universe
    where date = date '{latest}' and in_universe
    order by adv desc limit {N_NAMES * 2}
""").pl()["ticker"].to_list()
tickstr = "('" + "','".join(names) + "')"

px_wide = (wh.sql(f"""
    select ticker, date, adj_close_total
    from main_staging.stg_prices_adjusted
    where adj_close_total is not null and ticker in {tickstr}
    order by date
""").pl()
    .pivot(values="adj_close_total", index="date", on="ticker")
    .sort("date").tail(TOTAL))
px_wide = px_wide[:, [c for c in px_wide.columns if c == "date" or px_wide[c].null_count() == 0]]
tickers = [c for c in px_wide.columns if c != "date"][:N_NAMES]
dates = px_wide["date"].to_numpy()[1:]                              # T dates for the returns
R = np.diff(np.log(px_wide.select(tickers).to_numpy()), axis=0)     # T x N log returns
print(f"panel {R.shape}, {dates[0]} -> {dates[-1]}")

## Fama-French factors

Free from the Kenneth French Data Library: the five factors (`Mkt-RF`, `SMB`,
`HML`, `RMW`, `CMA`) plus momentum, daily, in percent. The loader keeps only the
daily rows (an 8-digit date) and drops the annual section at the foot of the file.

In [ ]:
def french(url, valcols):
    """Download one Kenneth French daily factor file into a polars frame."""
    raw = urllib.request.urlopen(
        urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=30).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    txt = z.read(z.namelist()[0]).decode("latin-1")
    rows = []
    for ln in txt.splitlines():
        p = [x.strip() for x in ln.split(",")]
        if len(p) == len(valcols) + 1 and p[0].isdigit() and len(p[0]) == 8:
            rows.append([p[0]] + [float(x) / 100 for x in p[1:]])   # percent -> fraction
    return (pl.DataFrame(rows, schema=["d"] + valcols, orient="row")
            .with_columns(pl.col("d").str.strptime(pl.Date, "%Y%m%d"))
            .rename({"d": "date"}))


BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
ff5 = french(BASE + "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip",
             ["Mkt_RF", "SMB", "HML", "RMW", "CMA", "RF"])
mom = french(BASE + "F-F_Momentum_Factor_daily_CSV.zip", ["Mom"])
fac = ff5.join(mom, on="date", how="inner")
FCOLS = ["Mkt_RF", "SMB", "HML", "RMW", "CMA", "Mom"]

# Align to the panel dates. Recent sessions with no factor yet drop out.
aligned = pl.DataFrame({"date": dates}).with_row_index("i").join(fac, on="date", how="inner")
idx = aligned["i"].to_numpy()
Rf = R[idx]                                    # returns on sessions that have factors
F = aligned.select(FCOLS).to_numpy()           # T x K factor returns
rf = aligned["RF"].to_numpy()                  # daily riskfree
T, N = Rf.shape
print(f"factors {fac['date'].min()} -> {fac['date'].max()}")
print(f"aligned T={T} N={N} q=N/T={N/T:.2f} (dropped {len(dates) - T} recent sessions to the FF lag)")

## The time-series factor model

Regress each name's excess return on the factors over the window:

$r_{i,t} - r^f_t = \alpha_i + \sum_k \beta_{i,k}\, f_{k,t} + \varepsilon_{i,t}$

The betas $B$ (N x K) are the loadings, $\Omega$ is the covariance of the factor
returns, and the idiosyncratic variances are the residual variances. The risk
model is $\Sigma = B\,\Omega\,B^\top + \operatorname{diag}(\sigma^2_\varepsilon)$.

The residual is assumed cross-sectionally uncorrelated (a diagonal), which is the
model's main limitation: any common structure the six factors miss (industry, for
one) is dropped.

In [ ]:
def gmv(S):
    """Global minimum-variance weights, w = S^-1 1 / (1' S^-1 1)."""
    one = np.ones(S.shape[0])
    try:
        x = np.linalg.solve(S, one)
    except np.linalg.LinAlgError:
        x = np.linalg.solve(S + 1e-8 * np.trace(S) / S.shape[0] * np.eye(S.shape[0]), one)
    return x / (one @ x)


def ff_cov(Rw, Fw):
    """Fama-French risk model: B Omega B' + diag(idiosyncratic var). Returns (Sigma, B)."""
    F1 = np.column_stack([np.ones(len(Fw)), Fw])          # intercept + factors
    coef, *_ = np.linalg.lstsq(F1, Rw, rcond=None)        # (K+1, N)
    B = coef[1:].T                                        # (N, K) loadings
    resid = Rw - F1 @ coef
    rv = (resid ** 2).sum(0) / (len(Fw) - F1.shape[1])    # idiosyncratic variance
    return B @ np.cov(Fw, rowvar=False) @ B.T + np.diag(rv), B


Sig_ff, B = ff_cov(Rf, F)
print("average loading per factor (cross-sectional mean beta):")
for k, name in enumerate(FCOLS):
    print(f"  {name:7s} {B[:, k].mean():+.3f}")
print(f"\nmarket-beta average {B[:, 0].mean():.2f} (near 1 as it should be)")

fig = px.bar(x=FCOLS, y=np.abs(B).mean(0),
             title="Average absolute factor loading across the universe")
fig.update_layout(xaxis_title="", yaxis_title="mean |beta|", height=380)
fig.show()

## Comparison: statistical estimators

The same estimators as `covariance.ipynb` — sample, nonlinear shrinkage (QIS) and
the PCA statistical factor model — so the Fama-French model can be scored against
them on the same panel.

In [ ]:
def c_sample(Rw):
    return np.cov(Rw, rowvar=False)


def c_pca(Rw):
    """Statistical factor model: top-K eigen-directions + diagonal idiosyncratic."""
    S = np.cov(Rw, rowvar=False)
    v, _ = np.linalg.eigh(np.corrcoef(Rw, rowvar=False))
    q = Rw.shape[1] / Rw.shape[0]
    K = max(1, int((v > (1 + np.sqrt(q)) ** 2).sum()))
    vs, Vs = np.linalg.eigh(S); ix = np.argsort(vs)[::-1][:K]
    lr = Vs[:, ix] @ np.diag(vs[ix]) @ Vs[:, ix].T
    return lr + np.diag(np.clip(np.diag(S) - np.diag(lr), 1e-12, None))


def qis(Rw):
    """Ledoit-Wolf nonlinear shrinkage (QIS). Works for T>N and N>T."""
    Tn, Nn = Rw.shape; Xc = Rw - Rw.mean(0); n = Tn - 1; c = Nn / n
    S = (Xc.T @ Xc) / n; S = (S + S.T) / 2
    lam, u = np.linalg.eigh(S); lam = np.clip(lam, 0.0, None)
    h = (min(c ** 2, 1 / c ** 2) ** 0.35) / Nn ** 0.35
    inv = 1.0 / lam[max(1, Nn - n + 1) - 1:Nn]; m = inv.size
    Lj = np.tile(inv, (m, 1)).T; Lji = Lj - Lj.T
    den = Lji ** 2 + (Lj ** 2) * h ** 2
    th = np.mean(Lj * Lji / den, 0); ht = np.mean(Lj * Lj * h / den, 0); a2 = th ** 2 + ht ** 2
    if Nn <= n:
        d = 1.0 / ((1 - c) ** 2 * inv + 2 * c * (1 - c) * inv * th + c ** 2 * inv * a2)
    else:
        d = np.concatenate([np.repeat(1.0 / ((c - 1) * np.mean(inv)), Nn - n), 1.0 / (inv * a2)])
    d = d * (lam.sum() / d.sum())
    return (u * d) @ u.T

## Structure: conditioning and spectra

At N approaching or above T the sample matrix is singular; every regularised
method is well-conditioned. The Fama-French model is low rank by construction
(six factors), so its spectrum has six large eigenvalues over a flat
idiosyncratic floor.

In [ ]:
rows = []
for name, S in [("sample", c_sample(Rf)), ("fama_french", Sig_ff),
                ("qis", qis(Rf)), ("pca", c_pca(Rf))]:
    ev = np.linalg.eigvalsh(S)
    rows.append({"method": name, "condition_number": float(np.linalg.cond(S)),
                 "posdef": bool(ev.min() > 0), "min_eig": float(ev.min())})
pl.DataFrame(rows).sort("condition_number")

In [ ]:
fig = go.Figure()
for name, S in [("sample", c_sample(Rf)), ("fama_french", Sig_ff),
                ("qis", qis(Rf)), ("pca", c_pca(Rf))]:
    ev = np.sort(np.linalg.eigvalsh(S))[::-1]
    fig.add_trace(go.Scatter(y=np.clip(ev, 1e-12, None), mode="lines", name=name))
fig.update_yaxes(type="log", title="eigenvalue (log)")
fig.update_layout(title="Covariance eigenvalue spectra", xaxis_title="rank", height=440)
fig.show()

## Economic test: out-of-sample minimum-variance risk

Estimate on a trailing window, form the global minimum-variance portfolio, hold
it forward, and score the realised volatility of the out-of-sample returns.
`EST_WIN=252` puts q = N/win well above 1, the regime that separates the methods.

In [ ]:
def backtest(kind, win, step=21):
    oos = []
    for d in range(win, T - 1, step):
        Rw = Rf[d - win:d]
        if kind == "fama_french":
            S = ff_cov(Rw, F[d - win:d])[0]
        else:
            S = {"sample": c_sample, "qis": qis, "pca": c_pca}[kind](Rw)
        oos.append(Rf[d:min(d + step, T)] @ gmv(S))
    return float(np.concatenate(oos).std() * np.sqrt(252) * 100)


EST_WIN = 252
res = pl.DataFrame([{"method": k, "oos_vol_pct": round(backtest(k, EST_WIN), 2)}
                   for k in ("fama_french", "qis", "pca", "sample")]).sort("oos_vol_pct")
print(f"EST_WIN={EST_WIN}  q=N/win={N / EST_WIN:.1f}")
res

In [ ]:
plot = res.filter(pl.col("oos_vol_pct") < 100)          # drop the sample blow-up for the scale
fig = px.bar(plot.to_pandas(), x="method", y="oos_vol_pct",
             title=f"Out-of-sample GMV volatility (EST_WIN={EST_WIN}); sample omitted, off scale")
fig.update_layout(yaxis_title="annualised vol %", xaxis_title="", height=420)
fig.show()

## Takeaways

- **Fama-French is the interpretable risk model.** Six named factors, loadings
  from returns alone, no characteristic data. It is positive-definite and
  well-conditioned even when N>T, and its factor covariance and loadings give risk
  attribution a statistical model cannot.
- **It is coarser than the statistical models on pure variance.** Six factors plus
  a diagonal residual miss the common structure QIS and PCA pick up, so its
  out-of-sample GMV volatility is higher than theirs, though far below the sample
  matrix. The diagonal-residual assumption is the reason; a residual-shrinkage step
  (POET-style) would close much of the gap.
- **Use it for attribution and hedging, the statistical models for the optimiser.**
  A fundamental (Barra) model with real characteristic loadings (size, value,
  industry) is the next step, and needs the characteristic data the roadmap defers
  to the ticker-details and fundamentals ingests.

The Fama-French factors lag live data by about two months, so a live risk model
uses them for the loadings and a statistical model for the freshest sessions.